In [11]:
# !pip install transformers
!pip install torch
!pip install scikit-learn

In [7]:
import unicodedata 
from typing import List, Tuple
from difflib import SequenceMatcher

In [9]:
def evaluate_encoding_accuracy(original: str, encoded: bytes, encoding: str = "utf-8") -> dict:
  decoded = encoded.decode(encoding, errors="replace")

  exact_match = original == decoded
  similarity = SequenceMatcher(None, original, decoded).ratio()

  byte_length = len(encoded)
  return {
          "exact_match": exact_match,
          "similarity_score": round(similarity, 4),
          "original_length": len(original),
          "byte_length": byte_length,
          "emoji_count": sum(1 for c in original if unicodedata.category(c).startswith('So'))
  }

In [10]:
def encode_text(text: str, encoding: str = "utf-8") -> bytes:
  try:
    encoded = text.encode(encoding, errors='replace')
    print(f"Encode successfully ({encoding}): {len(encoded)} bytes")
    return encoded
  except UnicodeEncodeError as e:
    print(f"Encode error: {e}. Try again with UTF-8")
    return text.encode("utf-8", errors="replace")

def decode_text(encoded_bytes: bytes, encoding: str = "utf-8") -> str:
  return encoded_bytes.decode(encoding, errors="replace")

text = "Xin chào! 😊🌍 Tôi là Quân, đang ở TP.HCM. #AI ❤️👨‍💻"
print("Câu gốc:", text)

bytes_data = encode_text(text)

decoded = decode_text(bytes_data)
print("Decode lại:", decoded)
print("Checking the similarity between encode and decode: ", text == decoded)  
print(evaluate_encoding_accuracy(text, bytes_data ))

emoji_complex = "👨‍👩‍👧‍👦🏽"  
print("Complex emoji:", emoji_complex)
print("Code points:", [hex(ord(c)) for c in emoji_complex])

Câu gốc: Xin chào! 😊🌍 Tôi là Quân, đang ở TP.HCM. #AI ❤️👨‍💻
Encode successfully (utf-8): 75 bytes
Decode lại: Xin chào! 😊🌍 Tôi là Quân, đang ở TP.HCM. #AI ❤️👨‍💻
Checking the similarity between encode and decode:  True
{'exact_match': True, 'similarity_score': 1.0, 'original_length': 50, 'byte_length': 75, 'emoji_count': 5}
Complex emoji: 👨‍👩‍👧‍👦🏽
Code points: ['0x1f468', '0x200d', '0x1f469', '0x200d', '0x1f467', '0x200d', '0x1f466', '0x1f3fd']


In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
tokenizer.add_tokens(["😊", "🌍", "❤️"])  

tokens = tokenizer.encode("Hello 😊", add_special_tokens=True)
print("Tokens:", tokens)
print("Decode again:", tokenizer.decode(tokens))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokens: [101, 31178, 119547, 102]
Decode lại: [CLS] Hello 😊 [SEP]


# How emoji attack the complexity of machine

In [2]:
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
import tiktoken
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def analyze_tokenization(text_no_emoji: str, text_with_emoji: str):
  print("=== TOKENIZATION COMPARISON ===")

  enc = tiktoken.get_encoding("cl100k_base")
  tokens_no = enc.encode(text_no_emoji)
  tokens_yes = enc.encode(text_with_emoji)

  print(f"GPT-4o tokenizer:")
  print(f"  No emoji: {len(tokens_no)} tokens → {tokens_no}")
  print(f"  With emoji   : {len(tokens_yes)} tokens → {tokens_yes}")
  print(f"  Increase: +{len(tokens_yes)-len(tokens_no)} tokens ({(len(tokens_yes)/len(tokens_no)-1)*100:.1f}%)")

  tok_bert = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
  t_no_bert = tok_bert.encode(text_no_emoji)
  t_yes_bert = tok_bert.encode(text_with_emoji)
  print(f"\nBERT multilingual:")
  print(f"  No emoji: {len(t_no_bert)-2} tokens")  
  print(f"  With emoji   : {len(t_yes_bert)-2} tokens")
  print(f"  Increase: +{len(t_yes_bert)-len(t_no_bert)} tokens")

def evaluate_embedding_quality(text_no: str, text_yes: str):
  print("\n=== EMBEDDING QUALITY ===")
  model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')  # model hỗ trợ emoji khá tốt
  
  emb_no = model.encode(text_no, convert_to_tensor=True).unsqueeze(0)
  emb_yes = model.encode(text_yes, convert_to_tensor=True).unsqueeze(0)
  
  sim = cosine_similarity(emb_no.cpu().numpy(), emb_yes.cpu().numpy())[0][0]
  print(f"Cosine similarity between two sentences: {sim:.4f}")
  print(f"→ IF < 0.85: embedding is seriously distort (impact on machine learning strongly)")
  
  # Độ phức tạp embedding
  print(f"Embedding dimension: {emb_no.shape[1]}")
  print(f"Variance of embedding (if high then complex):")
  print(f"  No emoji: {emb_no.var().item():.6f}")
  print(f"  With emoji   : {emb_yes.var().item():.6f}")

In [3]:
text_no_emoji = "Hôm nay thời tiết rất đẹp và tôi rất vui."
text_with_emoji = "Hôm nay thời tiết rất đẹp và tôi rất vui 😊🌞❤️"

analyze_tokenization(text_no_emoji, text_with_emoji)
evaluate_embedding_quality(text_no_emoji, text_with_emoji)

=== TOKENIZATION COMPARISON ===
GPT-4o tokenizer:
  No emoji: 24 tokens → [39, 95465, 308, 352, 270, 77394, 72, 9165, 52680, 436, 52458, 15199, 6655, 117, 79, 48842, 259, 9769, 72, 436, 52458, 348, 2005, 13]
  With emoji   : 31 tokens → [39, 95465, 308, 352, 270, 77394, 72, 9165, 52680, 436, 52458, 15199, 6655, 117, 79, 48842, 259, 9769, 72, 436, 52458, 348, 2005, 27623, 232, 9468, 234, 252, 49633, 97, 31643]
  Increase: +7 tokens (29.2%)

BERT multilingual:
  No emoji: 13 tokens
  With emoji   : 13 tokens
  Increase: +0 tokens

=== EMBEDDING QUALITY ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cosine similarity between two sentences: 0.9071
→ IF < 0.85: embedding is seriously distort (impact on machine learning strongly)
Embedding dimension: 384
Variance of embedding (if high then complex):
  No emoji: 0.060146
  With emoji   : 0.061314


In [4]:
def calculate_ml_complexity(seq_len_no: int, seq_len_yes: int, model_dim: int = 768):
  print("\n=== THE COMPLEXITY OF MACHINE LEARNING ===")

  cost_no = (seq_len_no ** 2) * model_dim
  cost_yes = (seq_len_yes ** 2) * model_dim
  increase = (cost_yes / cost_no - 1) * 100

  print(f"Sequence length increases from {seq_len_no} to {seq_len_yes}")
  print(f"Attention FLOPs increases: +{increase:.1f}%")
  print(f"Memory (probability) increases ~ {seq_len_yes/seq_len_no:.2f}x")
  print("→ With batch size 32, training time could increase 30–80%")

In [6]:
enc = tiktoken.get_encoding("cl100k_base")
seq_len_no = len(enc.encode(text_no_emoji))
seq_len_yes = len(enc.encode(text_with_emoji))
calculate_ml_complexity(seq_len_no, seq_len_yes)


=== THE COMPLEXITY OF MACHINE LEARNING ===
Sequence length increases from 24 to 31
Attention FLOPs increases: +66.8%
Memory (probability) increases ~ 1.29x
→ With batch size 32, training time could increase 30–80%
